# Tokyo Synthetic EVRP Scenario and Constraint-Gap Analysis

> 東京都の公開地理データとsynthetic customer configurationsを用いて、EV配送における各制約が、顧客の空間配置および分析条件の変化に対してどの程度拘束的になるかを探索的に評価する。

```text
公開データ
↓
synthetic scenario
↓
route proxy
↓
制約評価
↓
顧客配置・パラメータ感度
↓
量子VRP研究とのギャップ
↓
今後の研究優先課題
```

本分析は東京都の実配送失敗率、EVRP最適解、実際の充電行動、充電需要、充電器利用率、電力需要を推定しない。充電関連の評価は、候補地点への地理的アクセス、単純化した航続距離補完可能性、および一定出力による充電時間上限のproxyに限定する。


## 0. Research Question and Scope

**Purpose**  
研究上の位置づけ、対象、非対象、および成果の論理を固定する。

**Research question**  
Which EV-delivery constraints become binding across synthetic customer configurations and analytical conditions?

**Input**  
東京都公開データのprocessed proxies、synthetic assumptions、local quantum-VRP extraction notes。

**Method**  
探索的なscenario/route-proxy/constraint-gap分析として研究境界を明示する。

**Output**  
研究質問、分析フロー、valid/invalid interpretationの境界。

**Assumptions**  
顧客はsynthetic、routeはproxy、結果はモデル条件付き。

**What this step can show**  
どの制約・仮定・evidence gapを次に研究すべきか。

**What this step cannot show**  
東京都の実配送失敗率、最適ルート、実際の充電・電力需要。


In [1]:
from IPython.display import Markdown, display

display(Markdown(
    "**Primary research question:** Under explicit synthetic assumptions, which constraints are binding, "
    "which vary with customer configurations, and which remain insufficiently covered in the reviewed quantum-VRP evidence?"
))

**Primary research question:** Under explicit synthetic assumptions, which constraints are binding, which vary with customer configurations, and which remain insufficiently covered in the reviewed quantum-VRP evidence?

## 1. Environment and Configuration

**Purpose**  
repository root、依存環境、再現モード、ログ保存、実行状態を一元管理する。

**Research question**  
Can the analysis be executed without a machine-specific path or stale notebook state?

**Input**  
Current kernel, `.git` marker, `00_project_management/001_20260712_requirements.txt`, environment variable `TOKYO_EVRP_REPRODUCE`。

**Method**  
`.git`からrootを探索し、package versionsを収集し、subprocess stdout/stderrを保存する。

**Output**  
Environment table、execution configuration、step status。

**Assumptions**  
Full regeneration is destructive/time-consuming and therefore opt-in。

**What this step can show**  
実行環境と、どのモードで成果物が作られたか。

**What this step cannot show**  
異なるOS/package combinationでの完全同一性能。


In [2]:
from __future__ import annotations

import importlib.metadata as metadata
import os
import platform
import subprocess
import sys
import time
import warnings
from datetime import datetime, timezone
from pathlib import Path

warnings.filterwarnings("ignore", message="Pandas requires version.*numexpr")
warnings.filterwarnings("ignore", message="Pandas requires version.*bottleneck")

_start = Path.cwd().resolve()
_bootstrap_root = next(
    (candidate for candidate in (_start, *_start.parents) if (candidate / ".git").exists()),
    None,
)
if _bootstrap_root is None:
    raise RuntimeError(f"Repository root could not be found from {_start}.")
SCRIPTS_HINT = _bootstrap_root / "scripts"
if str(SCRIPTS_HINT) not in sys.path:
    sys.path.insert(0, str(SCRIPTS_HINT))

from validation_utils import (
    find_repository_root,
    generate_output_manifest,
    read_csv_checked,
    validate_output_manifest,
    write_csv_atomic,
)

ROOT = find_repository_root(Path.cwd())
SCRIPTS = ROOT / "scripts"
if str(SCRIPTS) not in sys.path:
    sys.path.insert(0, str(SCRIPTS))

REPRODUCE_ALL = os.environ.get("TOKYO_EVRP_REPRODUCE", "0") == "1"
RUN_SCRIPTS = REPRODUCE_ALL  # default False; explicit reproduction mode enables all stages
BOOTSTRAP_ITERATIONS = 1000
SEED_COUNT = 100
BOOTSTRAP_RANDOM_SEED = 20260711
LOG_DIR = ROOT / "outputs/logs"
LOG_DIR.mkdir(parents=True, exist_ok=True)
STEP_STATUS: list[dict[str, object]] = []

def run_step(step_name: str, command: list[str], timeout_seconds: int = 1200) -> None:
    'Run one command, persist stdout/stderr, and fail loudly on a non-zero return code.'
    started = time.perf_counter()
    completed = subprocess.run(
        command,
        cwd=ROOT,
        capture_output=True,
        text=True,
        timeout=timeout_seconds,
        check=False,
    )
    safe_name = step_name.lower().replace(" ", "_").replace("/", "_")
    stdout_path = LOG_DIR / f"{safe_name}.stdout.log"
    stderr_path = LOG_DIR / f"{safe_name}.stderr.log"
    stdout_path.write_text(completed.stdout, encoding="utf-8")
    stderr_path.write_text(completed.stderr, encoding="utf-8")
    status = {
        "step_name": step_name,
        "status": "success" if completed.returncode == 0 else "failed",
        "return_code": completed.returncode,
        "duration_seconds": time.perf_counter() - started,
        "stdout_log": str(stdout_path.relative_to(ROOT)),
        "stderr_log": str(stderr_path.relative_to(ROOT)),
        "warning": completed.stderr[-2000:] if completed.returncode == 0 else "",
        "error": completed.stderr[-2000:] if completed.returncode else "",
    }
    STEP_STATUS.append(status)
    print(completed.stdout)
    if completed.stderr:
        print(completed.stderr, file=sys.stderr)
    if completed.returncode != 0:
        raise RuntimeError(f"Step {step_name!r} failed with return code {completed.returncode}.")

packages = [
    "pandas", "numpy", "scipy", "scikit-learn", "matplotlib", "geopandas",
    "shapely", "networkx", "jupyter", "jupyterlab", "notebook", "nbformat",
    "nbclient", "seaborn", "numexpr", "bottleneck",
]
environment = [{"component": "Python", "version": sys.version.split()[0]}]
environment += [{"component": "OS", "version": platform.platform()}]
for package in packages:
    try:
        version = metadata.version(package)
    except metadata.PackageNotFoundError:
        version = "Not installed"
    environment.append({"component": package, "version": version})
environment_df = __import__("pandas").DataFrame(environment)
display(environment_df)
print(f"Repository root: {ROOT}")
print(f"REPRODUCE_ALL={REPRODUCE_ALL}; RUN_SCRIPTS={RUN_SCRIPTS}")

,component,version
0,Python,3.11.8
1,OS,macOS-14.5-arm64-arm-64bit
2,pandas,3.0.2
3,numpy,1.26.4
4,scipy,1.11.4
5,scikit-learn,1.7.2
6,matplotlib,3.10.7
7,geopandas,1.1.1
8,shapely,2.0.6
9,networkx,3.1


Repository root: /Users/tstakuma/Desktop/github/research
REPRODUCE_ALL=True; RUN_SCRIPTS=True


## 2. Input Data Inventory and Validation

**Purpose**  
存在だけでなく、bytes・header・schema・rows・numeric types・keysを検証する。

**Research question**  
Are canonical processed inputs readable and structurally valid?

**Input**  
Population mesh、OCM connections、depot/vehicle snapshots、parameter/evidence registries。

**Method**  
共通`read_csv_checked`とSHA-256 provenance registryで検証する。

**Output**  
Input preview; full inventory is written by the analysis pipeline。

**Assumptions**  
Raw archives may be macOS dataless placeholders; validated processed fallback is permitted and recorded。

**What this step can show**  
missing/zero-byte/header-only/bad schema/type/keyを区別できる。

**What this step cannot show**  
raw public-data archivesからの再download・再前処理が常に可能であること。


In [3]:
input_specs = {
    "population_mesh": (ROOT / "03_data/processed/413_20260705_estat_tokyo_mesh_population_cells.csv", ["mesh_code", "total_population"]),
    "charger_connections": (ROOT / "03_data/processed/428_20260705_open_charge_map_tokyo_boundary_clipped_connections.csv", ["connection_id", "latitude", "longitude"]),
    "depot_candidates": (ROOT / "03_data/processed/evrp_constraint_gap_inputs/419_20260711_depot_candidates_public_proxy_snapshot.csv", ["scenario_depot_id", "latitude", "longitude"]),
    "vehicle_specs": (ROOT / "03_data/processed/evrp_constraint_gap_inputs/423_20260711_vehicle_specs_public_source_snapshot.csv", ["scenario_vehicle_id", "battery_kwh", "catalog_range_km"]),
    "quantum_evidence": (ROOT / "03_data/processed/evrp_constraint_gap_inputs/421_20260711_quantum_vrp_evidence_registry.csv", ["reference_id", "paper_title", "page_or_section"]),
}
input_preview = []
for name, (path, required) in input_specs.items():
    frame = read_csv_checked(path, required_columns=required, require_nonempty=True)
    input_preview.append({
        "input": name,
        "path": str(path.relative_to(ROOT)),
        "rows": len(frame),
        "columns": len(frame.columns),
        "csv_state": frame.attrs["csv_state"],
        "max_missing_rate": float(frame.isna().mean().max()),
        "duplicate_rows": int(frame.duplicated().sum()),
    })
input_preview_df = __import__("pandas").DataFrame(input_preview)
display(input_preview_df)

,input,path,rows,columns,csv_state,max_missing_rate,duplicate_rows
0,population_mesh,03_data/processed/estat_tokyo_mesh_population_cel...,5448,5,data,0.942364,0
1,charger_connections,03_data/processed/open_charge_map_tokyo_boundary_...,137,18,data,0.773723,0
2,depot_candidates,03_data/processed/evrp_constraint_gap_inputs/depo...,440,12,data,0.000000,0
3,vehicle_specs,03_data/processed/evrp_constraint_gap_inputs/vehi...,10,16,data,0.200000,0
4,quantum_evidence,03_data/processed/evrp_constraint_gap_inputs/quan...,7,38,data,0.000000,0


## 3. Open-data Preprocessing

**Purpose**  
processed fallbackを明示し、全CSV分析を一度だけ実行する。

**Research question**  
Can the requested analysis be regenerated from validated local inputs without using stale outputs?

**Input**  
Section 2のvalidated inputs。

**Method**  
明示的再現モードではCSV-only analysis scriptを一回実行し、旧canonical CSV dirsを先に削除する。

**Output**  
`03_data/processed/**`、`06_outputs/tables/csv/**`、analysis manifest、step statuses。

**Assumptions**  
Raw-to-processed regeneration and road-network improved mode are unavailable; baseline fallback is explicit。

**What this step can show**  
新しいrun IDに属する完全なCSV分析成果物。

**What this step cannot show**  
raw source hydrationやOSM shortest pathの成功。


In [4]:
analysis_manifest_path = ROOT / "06_outputs/reports/validation/analysis_output_902_20260711_v04_manifest.csv"
if RUN_SCRIPTS:
    run_step(
        "analysis_csv_generation",
        [
            sys.executable,
            str(ROOT / "05_src/constraint_evaluation/run_tokyo_synthetic_evrp_analysis.py"),
            "--reproduce",
            "--seed-count", str(SEED_COUNT),
            "--bootstrap-iterations", str(BOOTSTRAP_ITERATIONS),
            "--bootstrap-random-seed", str(BOOTSTRAP_RANDOM_SEED),
        ],
    )
else:
    if not analysis_manifest_path.exists():
        raise RuntimeError(
            "Canonical outputs are absent. Re-run with TOKYO_EVRP_REPRODUCE=1 for explicit full regeneration."
        )
    STEP_STATUS.append({"step_name": "analysis_csv_generation", "status": "validated_existing", "return_code": 0, "duration_seconds": 0.0, "stdout_log": "Not run", "stderr_log": "Not run", "warning": "", "error": ""})

preprocessing_status = read_csv_checked(
    ROOT / "03_data/processed/scenario/444_20260711_open_data_preprocessing_status.csv",
    required_columns=["step", "mode", "raw_source_status", "result", "limitation"],
    require_nonempty=True,
)
display(preprocessing_status)

[2. Input Data Inventory and Validation] success (0.15s)
[3. Open-data Preprocessing] success (0.00s)
[4. Scenario Definition] success (0.05s)
[5. Synthetic Customer Generation] success (0.78s)
[6. Route Proxy Construction] success (25.98s)
[7. Constraint Evaluation] success (20.23s)
[8. Monte Carlo and Sensitivity Analysis] success (34.26s)
[10. Quantum VRP Evidence Comparison] success (0.05s)
[11. Research Presentation Summary Tables] success (0.02s)
[13. Validation and Export Summary] success (0.01s)
Analysis complete: 36 CSV artifacts
Run ID: 20260711T072949Z
Charging pathway generated: False
Charging-demand estimation generated: False
Charging-event timeline generated: False
Grid-load estimation generated: False



/opt/anaconda3/lib/python3.11/site-packages/pandas/core/computation/expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.8.7' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
/opt/anaconda3/lib/python3.11/site-packages/pandas/core/arrays/masked.py:56: UserWarning: Pandas requires version '1.4.2' or newer of 'bottleneck' (version '1.3.7' currently installed).
  from pandas.core import (



,step,mode,raw_source_status,processed_input,result,limitation
0,Population mesh,Validated processed-data fallback,Data unavailable in reproducible run (macOS da...,03_data/processed/estat_tokyo_mesh_population_cel...,Readable and schema-validated,Raw-to-processed regeneration was not executed.
1,Charger candidates,Validated processed-data fallback,Data unavailable in reproducible run (macOS da...,03_data/processed/open_charge_map_tokyo_boundary_...,Readable and schema-validated,Candidate attributes have substantial missingn...
2,Road network improved mode,Baseline fallback,Data unavailable,Not evaluated,Haversine distance plus explicit road multipli...,No shortest-path geometry or network travel ti...


## 4. Scenario Definition

**Purpose**  
scenarioを未来物語ではなく分析条件の組合せとして定義する。

**Research question**  
What customer, vehicle, charger, time, speed, and route-proxy conditions were compared?

**Input**  
Validated charger candidates、one coherent vehicle row、factor levels。

**Method**  
3 customer counts × 3 vehicle counts × 3 charger conditionsのfactorial design。

**Output**  
27 unique scenario configurations with explicit policies and IDs。

**Assumptions**  
All operating parameters are evidence-labelled assumptions, not forecasts。

**What this step can show**  
customer count、vehicle count、customers/vehicle、およびcharger conditionの効果を分けて比較できる。

**What this step cannot show**  
東京都の将来配送需要やfleet composition。


In [5]:
scenario_configurations = read_csv_checked(
    ROOT / "03_data/processed/scenario/446_20260711_scenario_configurations.csv",
    required_columns=["scenario_id", "customer_count", "vehicle_count", "charger_condition", "usable_range_km"],
    unique_keys=["scenario_id"],
    require_nonempty=True,
)
display(scenario_configurations[[
    "scenario_id", "customer_count", "vehicle_count", "customers_per_vehicle",
    "charger_condition", "eligible_charger_candidate_count", "usable_range_km",
    "payload_capacity_kg", "operating_time_limit_min", "assumed_speed_kmh",
]].head(12))
print(f"Scenario configurations: {len(scenario_configurations)} (expected 27)")

,scenario_id,customer_count,vehicle_count,customers_per_vehicle,charger_condition,eligible_charger_candidate_count,usable_range_km,payload_capacity_kg,operating_time_limit_min,assumed_speed_kmh
0,C025_V01_CHG_conservative_SPEED25_T480,25,1,25.000000,conservative,12,81.2,2000.0,480.0,25.0
1,C025_V01_CHG_balanced_SPEED25_T480,25,1,25.000000,balanced,106,81.2,2000.0,480.0,25.0
2,C025_V01_CHG_broad_SPEED25_T480,25,1,25.000000,broad,106,81.2,2000.0,480.0,25.0
3,C025_V03_CHG_conservative_SPEED25_T480,25,3,8.333333,conservative,12,81.2,2000.0,480.0,25.0
4,C025_V03_CHG_balanced_SPEED25_T480,25,3,8.333333,balanced,106,81.2,2000.0,480.0,25.0
5,C025_V03_CHG_broad_SPEED25_T480,25,3,8.333333,broad,106,81.2,2000.0,480.0,25.0
6,C025_V05_CHG_conservative_SPEED25_T480,25,5,5.000000,conservative,12,81.2,2000.0,480.0,25.0
7,C025_V05_CHG_balanced_SPEED25_T480,25,5,5.000000,balanced,106,81.2,2000.0,480.0,25.0
8,C025_V05_CHG_broad_SPEED25_T480,25,5,5.000000,broad,106,81.2,2000.0,480.0,25.0
9,C050_V01_CHG_conservative_SPEED25_T480,50,1,50.000000,conservative,12,81.2,2000.0,480.0,25.0


Scenario configurations: 27 (expected 27)


## 5. Synthetic Customer Generation

**Purpose**  
何がsyntheticで、何をseedごとに変化させたかを保存する。

**Research question**  
How do results vary when population-weighted customer locations, demand, and service time change?

**Input**  
e-Stat mesh population weights、explicit synthetic distributions、100 seed identifiers。

**Method**  
locationsはpopulation-weighted sample、demand/serviceはdiscrete-uniform synthetic draws。

**Output**  
17,500 customer rows、300 customer-count-specific configurations、randomization registry。

**Assumptions**  
Customer locations are not observed orders; demand and service time are not operational measurements。

**What this step can show**  
特定配置依存とseedを変えて共通する傾向。

**What this step cannot show**  
実顧客、実貨物重量、実サービス時間、独立な2,700試行。


In [6]:
customers = read_csv_checked(
    ROOT / "03_data/processed/scenario/447_20260711_synthetic_customers.csv",
    required_columns=["customer_configuration_id", "seed", "customer_id", "latitude", "longitude", "demand_kg", "service_time_min", "evidence_status"],
    unique_keys=["customer_configuration_id", "customer_id"],
    require_nonempty=True,
)
randomization = read_csv_checked(
    ROOT / "03_data/processed/scenario/443_20260711_monte_carlo_randomization_registry.csv",
    required_columns=["seed", "customer_locations_randomized", "customer_demands_randomized", "charger_availability_randomized", "speed_randomized"],
    require_nonempty=True,
)
display(customers.head())
display(randomization.head())
print(f"Unique seed identifiers: {customers['seed'].nunique()}")
print(f"Customer-count-specific configurations: {customers['customer_configuration_id'].nunique()}")

,customer_configuration_id,customer_count,seed,customer_id,sampling_weight_source,mesh_code,mesh_population,latitude,longitude,demand_kg,service_time_min,time_window_start,time_window_end,customer_demand_source,customer_location_source,evidence_status
0,C025_S001,25,1,C025_S001_N001,e-Stat total_population by mesh cell,533935261,6887.0,35.602083,139.703125,15,10,0.0,480.0,Synthetic discrete-uniform integer 5-30 kg ass...,Population-weighted synthetic sample; not obse...,Synthetic data with assumed parameters
1,C025_S001,25,1,C025_S001_N002,e-Stat total_population by mesh cell,533932952,3628.0,35.660417,139.321875,22,8,0.0,480.0,Synthetic discrete-uniform integer 5-30 kg ass...,Population-weighted synthetic sample; not obse...,Synthetic data with assumed parameters
2,C025_S001,25,1,C025_S001_N003,e-Stat total_population by mesh cell,533933833,2454.0,35.656250,139.415625,19,10,0.0,480.0,Synthetic discrete-uniform integer 5-30 kg ass...,Population-weighted synthetic sample; not obse...,Synthetic data with assumed parameters
3,C025_S001,25,1,C025_S001_N004,e-Stat total_population by mesh cell,533945521,7770.0,35.710417,139.653125,23,7,0.0,480.0,Synthetic discrete-uniform integer 5-30 kg ass...,Population-weighted synthetic sample; not obse...,Synthetic data with assumed parameters
4,C025_S001,25,1,C025_S001_N005,e-Stat total_population by mesh cell,533945592,6114.0,35.710417,139.746875,30,8,0.0,480.0,Synthetic discrete-uniform integer 5-30 kg ass...,Population-weighted synthetic sample; not obse...,Synthetic data with assumed parameters


,customer_configuration_id,seed,customer_count,customer_locations_randomized,customer_demands_randomized,service_times_randomized,vehicle_assignment_randomized,charger_availability_randomized,speed_randomized,pairing_rule,analysis_name
0,C025_S001,1,25,True,True,True,True,False,False,Seed identifier retained across all vehicle-co...,Monte Carlo spatial sensitivity analysis
1,C025_S002,2,25,True,True,True,True,False,False,Seed identifier retained across all vehicle-co...,Monte Carlo spatial sensitivity analysis
2,C025_S003,3,25,True,True,True,True,False,False,Seed identifier retained across all vehicle-co...,Monte Carlo spatial sensitivity analysis
3,C025_S004,4,25,True,True,True,True,False,False,Seed identifier retained across all vehicle-co...,Monte Carlo spatial sensitivity analysis
4,C025_S005,5,25,True,True,True,True,False,False,Seed identifier retained across all vehicle-co...,Monte Carlo spatial sensitivity analysis


Unique seed identifiers: 100
Customer-count-specific configurations: 300


## 6. Route Proxy Construction

**Purpose**  
customer assignment、visit order、distance definitionを追跡可能にする。

**Research question**  
What does the route proxy represent, and how does its distance vary?

**Input**  
Synthetic customers、public logistics-facility depot candidates、customer/vehicle factors。

**Method**  
KMeansで車両別割当てを近似し、nearest neighborで訪問順を近似する。

**Output**  
Base routes、route members、proxy edges、haversine/road-adjusted/network fields。

**Assumptions**  
Road multiplier=1.25 baseline; network distance is Data unavailable。

**What this step can show**  
仮訪問順序、顧客数/route、proxy distance distribution。

**What this step cannot show**  
EVRP最適解、時間窓/SOC最適化、道路上の実経路。


In [7]:
routes = read_csv_checked(
    ROOT / "03_data/processed/route_proxy/440_20260711_route_proxy_results.csv",
    required_columns=["scenario_route_proxy_id", "base_route_proxy_id", "haversine_distance_km", "road_adjusted_distance_km", "network_distance_km", "route_proxy_distance_km", "route_proxy_limitation"],
    unique_keys=["scenario_route_proxy_id"],
    require_nonempty=True,
)
distance_summary = read_csv_checked(
    ROOT / "03_data/processed/route_proxy/437_20260711_route_distance_summary.csv",
    required_columns=["mean", "median", "standard_deviation", "percentile_5", "percentile_95", "maximum", "distance_unmet_rate"],
    require_nonempty=True,
)
display(routes[["scenario_route_proxy_id", "customer_count_on_route", "route_proxy_distance_km", "route_total_demand_kg", "route_proxy_limitation"]].head())
display(distance_summary)

,scenario_route_proxy_id,customer_count_on_route,route_proxy_distance_km,route_total_demand_kg,route_proxy_limitation
0,C025_V01_CHG_balanced_SPEED25_T480__S001_R01,25,179.953053,464.0,Not an EVRP optimum; not time-window/SOC optim...
1,C025_V01_CHG_broad_SPEED25_T480__S001_R01,25,179.953053,464.0,Not an EVRP optimum; not time-window/SOC optim...
2,C025_V01_CHG_conservative_SPEED25_T480__S001_R01,25,179.953053,464.0,Not an EVRP optimum; not time-window/SOC optim...
3,C025_V03_CHG_balanced_SPEED25_T480__S001_R01,7,73.739203,133.0,Not an EVRP optimum; not time-window/SOC optim...
4,C025_V03_CHG_broad_SPEED25_T480__S001_R01,7,73.739203,133.0,Not an EVRP optimum; not time-window/SOC optim...


,metric,mean,median,standard_deviation,percentile_5,percentile_95,maximum,distance_unmet_rate,travel_cost_unmet_rate,interpretation
0,route_proxy_distance_km,122.370751,97.59148,86.48051,46.187201,327.93005,580.333428,Not applicable,Not applicable,Continuous route-proxy outcome; no distance or...


## 7. Constraint Evaluation

**Purpose**  
各未充足率のnumerator/denominatorと未評価項目を分離する。

**Research question**  
Which constraints became binding under the current synthetic assumptions?

**Input**  
Condition-route proxy results。

**Method**  
payload、operating time、range、candidate accessを全routeで評価し、assisted range/durationは適格routeだけを分母にする。

**Output**  
Long constraint evaluations、case rates、route-weighted summary、payload/time diagnostics。

**Assumptions**  
SOC trajectory、visit-level time windows、waiting/detour are not evaluated。

**What this step can show**  
モデル内の制約拘束性と定義別分母。

**What this step cannot show**  
実配送の失敗率、SOC feasibility、driver-hours compliance。


In [8]:
constraint_summary = read_csv_checked(
    ROOT / "03_data/processed/constraints/407_20260711_constraint_summary.csv",
    required_columns=["constraint_name", "route_weighted_unmet_rate", "case_weighted_unmet_rate", "evaluated_route_count", "numerator_definition", "denominator_definition", "evidence_status"],
    require_nonempty=True,
)
display(constraint_summary[[
    "constraint_name", "route_weighted_unmet_rate", "case_weighted_unmet_rate",
    "confidence_interval_lower", "confidence_interval_upper", "evaluated_route_count",
    "spatial_sensitivity", "evidence_status",
]])
display(read_csv_checked(ROOT / "03_data/processed/constraints/409_20260711_payload_diagnostics.csv", require_nonempty=True))
display(read_csv_checked(ROOT / "03_data/processed/constraints/412_20260711_time_constraint_diagnostics.csv", require_nonempty=True))

,constraint_name,route_weighted_unmet_rate,case_weighted_unmet_rate,confidence_interval_lower,confidence_interval_upper,evaluated_route_count,spatial_sensitivity,evidence_status
0,Payload capacity,0.000000,0.000000,0.000000,0.000000,8100,Low,Synthetic data with assumed parameters
1,Operating-time limit,0.335185,0.529556,0.327778,0.343333,8100,Low,Synthetic route proxy with assumed parameters
2,Range feasibility,0.644444,0.767259,0.631111,0.657407,8100,Medium,Manufacturer range with assumed usable ratio a...
3,SOC feasibility,NaN,NaN,NaN,NaN,0,Not assessable,Not evaluated
4,Charging-station access,0.106914,0.076691,0.100123,0.113580,8100,Low,Public candidate geography with substantial at...
5,Charging-assisted range feasibility,0.332950,0.469855,0.321483,0.344649,5220,Medium,Simplified proxy with public candidate geography
6,Charging-duration feasibility,0.153021,0.286569,0.148042,0.157951,4849,Low,Simplified duration proxy; actual charging beh...


,customer_demand_distribution,customer_demand_min_kg,customer_demand_max_kg,customer_demand_mean_kg,customer_demand_standard_deviation_kg,route_demand_mean_kg,route_demand_max_kg,vehicle_payload_capacity_kg,maximum_payload_utilization_ratio,percentile_95_payload_utilization_ratio,evidence_status,interpretation_caution
0,Discrete uniform integer 5-30 kg (synthetic),5.0,30.0,17.575886,7.476294,341.753333,1935.0,2000.0,0.9675,0.4575,Synthetic data with assumed parameters,A zero unmet rate would mean only that this sy...


,factor,parameter,baseline_value,sensitivity_range,unmet_rate_impact,interpretation
0,Assumed travel speed,assumed_speed_kmh,25.0,20-30,0.496667,Exploratory one-at-a-time effect under the rou...
1,Road-distance multiplier,road_distance_multiplier,1.25,1.1-1.4,0.300000,Exploratory one-at-a-time effect under the rou...
2,Service time per customer,service_time_per_customer_min,10.0,5-15,0.606667,Exploratory one-at-a-time effect under the rou...
3,Operating-time limit,operating_time_limit_min,480.0,360-600,0.780000,Exploratory one-at-a-time effect under the rou...
4,Vehicle count,vehicle_count,3.0,1-5,0.992000,Exploratory one-at-a-time effect under the rou...
5,Customer count,customer_count,50.0,25-100,0.973333,Exploratory one-at-a-time effect under the rou...
6,Route-proxy inefficiency,route_generation_method,KMeans + nearest neighbor,Alternative route generators not evaluated,NaN,Not quantified; route proxy is not an optimize...
7,Supplemental charging duration,charging_time_total_min,0,Not included in operating-time constraint,NaN,Evaluated separately as a duration proxy; no c...
8,Waiting and charger detour time,waiting_time_total_min/detour_time_min,0,Not modeled,NaN,"Data unavailable; zero is a model boundary, no..."


## 8. Monte Carlo and Sensitivity Analysis

**Purpose**  
配置依存性、paired design、仮定感度を統計的に分離する。

**Research question**  
Which findings persist across seeds, and which are driven by analytical assumptions?

**Input**  
100 paired seed clusters、2,700 conditional cases、13 requested sensitivity levers。

**Method**  
seed-cluster bootstrap 1,000回とthree-level OAT sensitivity。

**Output**  
Route/case-weighted rates、95% CI、seed variability、sensitivity response。

**Assumptions**  
Bootstrap unit is seed; all conditions within a sampled seed are retained。

**What this step can show**  
配置変動とパラメータ応答の相対的重要性。

**What this step cannot show**  
2,700独立試行、causal effect、calibrated operational elasticities。


In [9]:
statistics = read_csv_checked(
    ROOT / "06_outputs/reports/validation/687_20260711_statistical_validation.csv", require_nonempty=True
)
sensitivity = read_csv_checked(
    ROOT / "03_data/processed/constraints/411_20260711_sensitivity_summary.csv",
    required_columns=["parameter", "level", "constraint_name", "route_weighted_unmet_rate", "unmet_rate_change_from_base"],
    require_nonempty=True,
)
display(statistics)
display(
    sensitivity.assign(abs_change=sensitivity["unmet_rate_change_from_base"].abs())
    .sort_values("abs_change", ascending=False)
    .head(15)
)

,analysis_run_id,independent_seed_count,customer_count_specific_configuration_count,scenario_configuration_count,conditional_evaluation_count,base_route_proxy_count,condition_route_evaluation_count,bootstrap_iterations,bootstrap_random_seed,monte_carlo_name
0,20260711T072949Z,100,300,27,2700,2700,8100,1000,20260711,Monte Carlo spatial sensitivity analysis


,parameter,level,parameter_value,constraint_name,route_weighted_unmet_rate,evaluated_route_count,unmet_route_count,customer_count_for_evaluation,vehicle_count_for_evaluation,charger_condition_for_evaluation,independent_seed_count,sensitivity_design,evidence_status,base_unmet_rate,unmet_rate_change_from_base,abs_change
77,vehicle_count,low,1.0,Charging-duration feasibility,1.000000,100,100,50,1,balanced,100,Three-level one-at-a-time paired-seed analysis,Exploratory parameter sensitivity; not calibra...,0.000000,1.000000,1.000000
76,vehicle_count,low,1.0,Charging-assisted range feasibility,1.000000,100,100,50,1,balanced,100,Three-level one-at-a-time paired-seed analysis,Exploratory parameter sensitivity; not calibra...,0.056738,0.943262,0.943262
73,vehicle_count,low,1.0,Operating-time limit,1.000000,100,100,50,1,balanced,100,Three-level one-at-a-time paired-seed analysis,Exploratory parameter sensitivity; not calibra...,0.383333,0.616667,0.616667
103,customer_count,high,100.0,Operating-time limit,0.990000,300,297,100,3,balanced,100,Three-level one-at-a-time paired-seed analysis,Exploratory parameter sensitivity; not calibra...,0.383333,0.606667,0.606667
86,vehicle_count,high,5.0,Range feasibility,0.412000,500,206,50,5,balanced,100,Three-level one-at-a-time paired-seed analysis,Exploratory parameter sensitivity; not calibra...,0.940000,-0.528000,0.528000
55,operating_time_limit_min,low,360.0,Operating-time limit,0.826667,300,248,50,3,balanced,100,Three-level one-at-a-time paired-seed analysis,Exploratory parameter sensitivity; not calibra...,0.383333,0.443333,0.443333
106,customer_count,high,100.0,Charging-assisted range feasibility,0.440000,300,132,100,3,balanced,100,Three-level one-at-a-time paired-seed analysis,Exploratory parameter sensitivity; not calibra...,0.056738,0.383262,0.383262
85,vehicle_count,high,5.0,Operating-time limit,0.008000,500,4,50,5,balanced,100,Three-level one-at-a-time paired-seed analysis,Exploratory parameter sensitivity; not calibra...,0.383333,-0.375333,0.375333
91,customer_count,low,25.0,Operating-time limit,0.016667,300,5,25,3,balanced,100,Three-level one-at-a-time paired-seed analysis,Exploratory parameter sensitivity; not calibra...,0.383333,-0.366667,0.366667
67,operating_time_limit_min,high,600.0,Operating-time limit,0.046667,300,14,50,3,balanced,100,Three-level one-at-a-time paired-seed analysis,Exploratory parameter sensitivity; not calibra...,0.383333,-0.336667,0.336667


## 9. Charger-access and Charging-assisted Feasibility

**Purpose**  
充電関連評価を候補アクセス・単純化したrange support・duration proxyに限定する。

**Research question**  
Is a screened charger candidate geographically near, and can a simplified support assumption cover range exceedance?

**Input**  
OCM candidate records、route-proxy nodes、condition-specific thresholds、reported power。

**Method**  
nearest candidate distance、two-usable-range rule、constant reported-power duration。

**Output**  
Candidate counts、nearest distance、geographic access、assisted-range/duration feasibility。

**Assumptions**  
Actual public access、availability、queue、failure、operating hours、arrival SOC、stop choice are unknown/not evaluated。

**What this step can show**  
設定した地理閾値と単純化条件で候補支援が可能か。

**What this step cannot show**  
実際の充電時刻・地点選択・量・需要・利用率・電力負荷。


In [10]:
charger_definitions = read_csv_checked(
    ROOT / "03_data/processed/charger_access/402_20260711_charger_condition_definitions.csv",
    required_columns=["charger_condition", "eligible_charger_candidate_count", "maximum_access_distance_km", "condition_definition"],
    require_nonempty=True,
)
charger_results = read_csv_checked(
    ROOT / "03_data/processed/charger_access/404_20260711_route_charger_access_results.csv",
    required_columns=["charger_geographically_accessible", "charger_arrival_soc_feasible", "charging_assisted_range_feasible", "charging_duration_feasible"],
    require_nonempty=True,
)
display(charger_definitions)
display(charger_results.head())
display(Markdown(
    "**Boundary:** this section does not infer an actual charging event, time series, station choice, demand, utilization, or grid load."
))

,charger_condition,minimum_power_kw,maximum_access_distance_km,usage_type_policy,missing_power_policy,missing_usage_type_policy,public_access_required,charging_time_limit_min,connector_compatibility_policy,operating_status_policy,condition_definition,eligible_charger_candidate_count,selected_charger_count,assistance_eligible_charger_count
0,conservative,50.0,3.0,Retain missing usage type and flag public acce...,Exclude,Retain and flag Unknown,False,60.0,Require reported CHAdeMO,Require reported Operational,Operational CHAdeMO connection with known powe...,12,12,12
1,balanced,40.0,5.0,Retain missing usage type and flag public acce...,Exclude,Retain and flag Unknown,False,60.0,Require reported CHAdeMO,Exclude only explicitly unavailable records; r...,CHAdeMO connection not explicitly unavailable ...,106,106,106
2,broad,NaN,10.0,Retain missing usage type and flag public acce...,Retain for geography; exclude from duration ev...,Retain and flag Unknown,False,90.0,Retain non-Tesla candidates; compatibility mus...,Exclude only explicitly unavailable records; r...,Non-Tesla candidate not explicitly unavailable...,106,106,106


,scenario_id,scenario_route_proxy_id,seed,customer_count,vehicle_count,charger_condition,charger_candidate_count,nearest_candidate_charger_id,nearest_charger_haversine_distance_km,nearest_charger_distance_km,...,charger_geographically_accessible,charger_public_access_known,charger_power_known,charger_connector_compatibility_known,charger_operating_status_known,charger_arrival_soc_feasible,charging_assisted_range_evaluated,charging_assisted_range_feasible,charging_duration_evaluated,charging_duration_feasible
0,C025_V01_CHG_balanced_SPEED25_T480,C025_V01_CHG_balanced_SPEED25_T480__S001_R01,1,25,1,balanced,106,17723_17964,0.423433,0.529291,...,True,False,True,True,False,Not evaluated,True,False,True,True
1,C025_V01_CHG_broad_SPEED25_T480,C025_V01_CHG_broad_SPEED25_T480__S001_R01,1,25,1,broad,106,17723_17964,0.423433,0.529291,...,True,False,True,True,False,Not evaluated,True,False,True,True
2,C025_V01_CHG_conservative_SPEED25_T480,C025_V01_CHG_conservative_SPEED25_T480__S001_R01,1,25,1,conservative,12,62645_79567,1.261104,1.576379,...,True,False,True,True,True,Not evaluated,True,False,True,True
3,C025_V03_CHG_balanced_SPEED25_T480,C025_V03_CHG_balanced_SPEED25_T480__S001_R01,1,25,3,balanced,106,17723_17964,0.423433,0.529291,...,True,False,True,True,False,Not evaluated,False,NaN,False,NaN
4,C025_V03_CHG_broad_SPEED25_T480,C025_V03_CHG_broad_SPEED25_T480__S001_R01,1,25,3,broad,106,17723_17964,0.423433,0.529291,...,True,False,True,True,False,Not evaluated,False,NaN,False,NaN


**Boundary:** this section does not infer an actual charging event, time series, station choice, demand, utilization, or grid load.

## 10. Quantum VRP Evidence Comparison

**Purpose**  
operational requirementsとreviewed quantum-VRP evidenceを報告粒度を保って比較する。

**Research question**  
Which operationally important requirements remain insufficiently covered in the reviewed quantum-VRP evidence?

**Input**  
Seven local extraction notes and normalized conservative registry。

**Method**  
reported facts only; missing resource/performance values are `Not reported`; route count/qubit width/hardware execution are not equated。

**Output**  
Paper-row evidence CSV and requirement gap table。

**Assumptions**  
This is a bounded evidence set, not a systematic review of all literature。

**What this step can show**  
reviewed evidence内のcoverageとreporting gap。

**What this step cannot show**  
未報告項目の推定、全量子VRP研究における不存在、量子優位性。


In [11]:
quantum_evidence = read_csv_checked(
    ROOT / "03_data/processed/quantum_gap/434_20260711_quantum_vrp_evidence.csv",
    required_columns=["reference_id", "paper_title", "authors", "year", "doi", "url", "page_or_section", "execution_type"],
    require_nonempty=True,
)
quantum_gap = read_csv_checked(
    ROOT / "06_outputs/tables/csv/690_20260711_table_03_quantum_vrp_gap.csv",
    required_columns=["evaluation_item", "quantum_vrp_coverage", "evidence_gap", "gap_basis", "reference_id"],
    require_nonempty=True,
)
display(quantum_evidence[["reference_id", "paper_title", "problem_type", "customer_count", "vehicle_count", "execution_type", "classical_baseline"]])
display(quantum_gap[["evaluation_item", "analysis_importance", "quantum_vrp_coverage", "evidence_gap", "gap_basis"]])

,reference_id,paper_title,problem_type,customer_count,vehicle_count,execution_type,classical_baseline
0,QVRP-AZAD-2020,Solving Vehicle Routing Problem Using Quantum ...,VRP,Problem tuples (4;2); (5;2); (5;3) are reporte...,2-3 in reported problem tuples; exact semantic...,Not reported in local extraction note,Not reported in local extraction note
1,QCVRP-PALACKAL-2023,Quantum-Assisted Solution Paths for the Capaci...,CVRP decomposed into clustering and TSP subpro...,Not reported,Not reported,Classical emulation/simulation implied by loca...,Not reported in local extraction note
2,QVRPTW-LEONIDAS-2023,Qubit efficient quantum algorithms for the veh...,VRPTW,Not reported,Not reported,Quantum simulator and quantum hardware,Gurobi comparison reported
3,QCVRP-XIE-2023,A Feasibility-Preserved Quantum Approximate So...,CVRP,Not reported,Not reported,Classical emulation/simulation implied by loca...,Not reported in local extraction note
4,QHVRP-FITZEK-2024,Applying quantum approximate optimization to t...,HVRP,3 customers,2 trucks,Quantum simulation,Not reported as a classical VRP solver baseline
5,QVRP-AZFAR-2025,Quantum-Assisted Vehicle Routing: Realizing QA...,VRP,Not reported,Not reported,Quantum hardware pipeline and simulator/hardwa...,Not reported
6,QCVRP-ONAH-2025,Requirements for Early Quantum Utility and Qua...,CVRP utility/resource assessment,Golden-5 example; customer count Not reported ...,Not reported,Resource estimate,Not reported in local extraction note


,evaluation_item,analysis_importance,quantum_vrp_coverage,evidence_gap,gap_basis
0,Customer scale,Directly varied in the factorial design,Reported in at least one reviewed study,High,Reviewed evidence reports small/ambiguous prob...
1,Multiple vehicles,Directly varied in the factorial design,Reported in at least one reviewed study,Medium,"Some reviewed coverage exists, but operational..."
2,Capacity,Current route-weighted unmet rate for Payload ...,Reported in at least one reviewed study,Medium,"Some reviewed coverage exists, but operational..."
3,Time windows,Current route-weighted unmet rate for Operatin...,Reported in at least one reviewed study,Medium,"Some reviewed coverage exists, but operational..."
4,Battery/SOC,Current route-weighted unmet rate for Range fe...,Not reported in the reviewed evidence set,High,Requirement matters in the synthetic analysis ...
5,Charging stations,Current route-weighted unmet rate for Charging...,Not reported in the reviewed evidence set,High,Requirement matters in the synthetic analysis ...
6,Charging duration,Current route-weighted unmet rate for Charging...,Not reported in the reviewed evidence set,High,Requirement matters in the synthetic analysis ...
7,Multi-depot,Not evaluated or not directly quantified in th...,Not reported in the reviewed evidence set,Not assessable,Current analysis does not quantify operational...
8,Heterogeneous vehicles,Not evaluated or not directly quantified in th...,Reported in at least one reviewed study,Not assessable,Current analysis does not quantify operational...
9,Dynamic demand,Not evaluated or not directly quantified in th...,Not reported in the reviewed evidence set,Not assessable,Current analysis does not quantify operational...


## 11. Research Presentation Summary Tables

**Purpose**  
分析CSVから研究発表用5表と17図を再現する。

**Research question**  
Can presentation claims be traced to unrounded analysis CSVs?

**Input**  
Canonical analysis CSVs only。

**Method**  
Rendererを一度だけ実行し、CSVの値は未丸め、PNG/SVGでのみ表示整形する。

**Output**  
6 table CSV/PNG/SVG (Table 5 full/slide) and 17 figure PNG/SVG/source-data CSV。

**Assumptions**  
Japanese font is selected from installed fonts; all figures are explanatory, not operational predictions。

**What this step can show**  
研究質問に対応する表・図とそのsource-data traceability。

**What this step cannot show**  
source CSVに存在しない固定数値や実運用の確定的結論。


In [12]:
render_manifest_path = ROOT / "06_outputs/reports/validation/render_output_902_20260711_v04_manifest.csv"
if RUN_SCRIPTS:
    run_step(
        "csv_driven_rendering",
        [sys.executable, str(ROOT / "05_src/visualization/render_tokyo_synthetic_evrp_outputs.py"), "--reproduce"],
    )
else:
    if not render_manifest_path.exists():
        raise RuntimeError(
            "Rendered outputs are absent. Re-run with TOKYO_EVRP_REPRODUCE=1 for explicit full regeneration."
        )
    STEP_STATUS.append({"step_name": "csv_driven_rendering", "status": "validated_existing", "return_code": 0, "duration_seconds": 0.0, "stdout_log": "Not run", "stderr_log": "Not run", "warning": "", "error": ""})

table_files = sorted((ROOT / "06_outputs/tables/csv").glob("table_*.csv"))
figure_files = sorted((ROOT / "06_outputs/06_outputs/figures/active/active/png").glob("figure_*.png"))
display(__import__("pandas").DataFrame({"table_csv": [str(path.relative_to(ROOT)) for path in table_files]}))
print(f"Table CSVs: {len(table_files)}; PNG figures: {len(figure_files)}")
display(read_csv_checked(ROOT / "06_outputs/tables/csv/691_20260711_table_04_integrated_research_summary.csv", require_nonempty=True))

Rendered 17 figures and 6 tables.
Japanese-capable font selection: Hiragino Sans



/opt/anaconda3/lib/python3.11/site-packages/pandas/core/computation/expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.8.7' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
/opt/anaconda3/lib/python3.11/site-packages/pandas/core/arrays/masked.py:56: UserWarning: Pandas requires version '1.4.2' or newer of 'bottleneck' (version '1.3.7' currently installed).
  from pandas.core import (



,table_csv
0,06_outputs/tables/csv/688_20260711_table_01_scenario_design.csv
1,06_outputs/tables/csv/table_02_constraint_unmet_v...
2,06_outputs/tables/csv/690_20260711_table_03_quantum_vrp_gap.csv
3,06_outputs/tables/csv/table_04_integrated_researc...
4,06_outputs/tables/csv/table_05_constraint_interpr...
5,06_outputs/tables/csv/table_05_constraint_interpr...


Table CSVs: 6; PNG figures: 17


,constraint_or_requirement,synthetic_scenario_definition,route_weighted_unmet_rate,confidence_interval,spatial_variability,parameter_sensitivity,maximum_parameter_response,main_assumption,evidence_quality,quantum_vrp_coverage,evidence_gap,research_priority,interpretation
0,Payload capacity,Routes whose synthetic route demand exceeds th...,0.000000,0.000000-0.000000,Low,Low,0.000000,Synthetic discrete-uniform customer demand and...,Synthetic data with assumed parameters,Reported in at least one reviewed study,Medium,Evidence-dependent,0% unmet under synthetic demand assumptions; c...
1,Operating-time limit,Route proxies whose travel plus service durati...,0.335185,0.327778-0.343333,Low,High,0.616667,"Haversine distance with road multiplier, assum...",Synthetic route proxy with assumed parameters,Reported in at least one reviewed study,Medium,Medium,"Merits further study, but inference remains li..."
2,Range feasibility,Route proxies whose estimated distance exceeds...,0.644444,0.631111-0.657407,Medium,High,0.528000,Nominal range multiplied by initial-minus-rese...,Manufacturer range with assumed usable ratio a...,Not reported in the reviewed evidence set,High,High,Binding or highly assumption-sensitive in the ...
3,SOC feasibility,Not evaluated,NaN,Not evaluated,Not assessable,Not assessable,NaN,A sequential SOC trajectory and charger-arriva...,Not evaluated,Not reported in the reviewed evidence set,High,Evidence-dependent,Constraint is not evaluated; implementation an...
4,Charging-station access,Route proxies with no retained candidate withi...,0.106914,0.100123-0.113580,Low,Low,0.000000,Distance from any route-proxy node to a screen...,Public candidate geography with substantial at...,Not reported in the reviewed evidence set,High,Medium,"Merits further study, but inference remains li..."
5,Charging-assisted range feasibility,Range-infeasible route proxies not supportable...,0.332950,0.321483-0.344649,Medium,High,0.943262,Simplified geographic range-support proxy; cha...,Simplified proxy with public candidate geography,Not reported in the reviewed evidence set,High,High,Binding or highly assumption-sensitive in the ...
6,Charging-duration feasibility,Evaluable range-infeasible route proxies whose...,0.153021,0.148042-0.157951,Low,High,1.000000,"Constant reported power; no taper, efficiency ...",Simplified duration proxy; actual charging beh...,Not reported in the reviewed evidence set,High,High,Binding or highly assumption-sensitive in the ...


## 12. Interpretation and Limitations

**Purpose**  
各数値のvalid interpretation、invalid interpretation、未評価項目を明示する。

**Research question**  
What can and cannot be claimed from each model-conditional result?

**Input**  
Table 5、assumptions/limitations registry、constraint evidence statuses。

**Method**  
Observed value、definition、assumption、evidence qualityを一体で表示する。

**Output**  
Full/slide interpretation tables and limitations registry。

**Assumptions**  
Low unmet rate is not evidence that a constraint is absent operationally。

**What this step can show**  
発表で言えることと言えないこと。

**What this step cannot show**  
synthetic/proxy evidenceをreal-world observationへ一般化すること。


In [13]:
interpretation_table = read_csv_checked(
    ROOT / "06_outputs/tables/csv/692_20260711_table_05_constraint_interpretation_full.csv",
    required_columns=["constraint_name", "observed_value", "valid_interpretation", "invalid_interpretation", "evidence_status"],
    require_nonempty=True,
)
limitations = read_csv_checked(
    ROOT / "03_data/processed/scenario/442_20260711_assumptions_and_limitations.csv",
    required_columns=["topic", "limitation", "status"],
    require_nonempty=True,
)
display(interpretation_table[["constraint_name", "observed_value", "evidence_status", "valid_interpretation", "invalid_interpretation"]])
display(limitations)

,constraint_name,observed_value,evidence_status,valid_interpretation,invalid_interpretation
0,Payload capacity,0.0,Synthetic data with assumed parameters,現在の需要仮定では積載量制約がどの程度拘束的だったか。,東京都のEV配送では積載量制約が存在しない。
1,Operating-time limit,0.3351851851851852,Synthetic route proxy with assumed parameters,現在の速度・時間上限・顧客数・route proxyの下で時間制約が拘束した。,東京都の実配送の同じ割合が時間内に終了しない。
2,Range feasibility,0.6444444444444445,Manufacturer range with assumed usable ratio a...,現在のvehicle rangeとroute proxyでは途中補完なしで成立しない候補がある。,実車両が同じ割合で電欠する。
3,SOC feasibility,Not evaluated,Not evaluated,SOC trajectoryは未評価である。,単純な距離閾値によりSOC feasibleである。
4,Charging-station access,0.10691358024691358,Public candidate geography with substantial at...,設定した地理的・条件的閾値で候補が見つかるか。,実運用でも確実に充電器を利用できる。
5,Charging-assisted range feasibility,0.33295019157088124,Simplified proxy with public candidate geography,単純化した充電補完モデルでも解消できない航続距離条件が残る。,実車両が特定の充電器を特定時刻に利用する。
6,Charging-duration feasibility,0.15302124149309135,Simplified duration proxy; actual charging beh...,一定出力の単純計算で補完時間が設定上限内か。,実際の充電時間・待ち時間・充電行動が予測できた。


,topic,limitation,status
0,Customer locations,Synthetic population-weighted locations; not o...,Assumption or evidence limitation
1,Customer demand,Synthetic 5-30 kg values; not observed freight...,Assumption or evidence limitation
2,Route construction,KMeans plus nearest-neighbor proxy; not an EVR...,Assumption or evidence limitation
3,Road geometry,Haversine plus multiplier baseline; road-netwo...,Assumption or evidence limitation
4,Travel speed,Assumed constant value; not calibrated traffic...,Assumption or evidence limitation
5,Service time,Synthetic value; not observed stop-service time,Assumption or evidence limitation
6,Visit time windows,Not evaluated,Not evaluated
7,SOC trajectory,Not evaluated,Not evaluated
8,Charger congestion,Not evaluated,Not evaluated
9,Charger failure,Not evaluated,Not evaluated


## 13. Validation and Export Summary

**Purpose**  
全工程、統計、出力、research integrityを最終検証する。

**Research question**  
Did every required stage and artifact complete in the current run without forbidden output classes?

**Input**  
Execution statuses、analysis/render manifests、canonical output directories、integrity checks。

**Method**  
hash/size/mtime/CSV shape検証、count assertions、forbidden filename scan、explicit false flags。

**Output**  
Final manifest、notebook status、validation summary。

**Assumptions**  
Deprecated outputs are excluded from active validation。

**What this step can show**  
Run All success/failure、current artifact completeness、research-scope integrity。

**What this step cannot show**  
deprecated historical artifactsが正しいこと、外部data hydration。


In [14]:
analysis_manifest = read_csv_checked(
    analysis_manifest_path,
    required_columns=["analysis_run_id", "path", "size_bytes", "sha256", "row_count", "column_count", "status"],
    require_nonempty=True,
)
render_manifest = read_csv_checked(
    render_manifest_path,
    required_columns=["path", "size_bytes", "sha256", "status"],
    require_nonempty=True,
)
validate_output_manifest(analysis_manifest.drop(columns=["analysis_run_id"]), root=ROOT)
validate_output_manifest(render_manifest, root=ROOT)

canonical_dirs = [
    ROOT / "outputs/data",
    ROOT / "outputs/tables",
    ROOT / "outputs/figures",
    ROOT / "outputs/logs",
    ROOT / "outputs/validation",
]
final_manifest_path = ROOT / "06_outputs/reports/validation/final_output_902_20260711_v04_manifest.csv"
final_manifest = generate_output_manifest(
    canonical_dirs,
    manifest_path=final_manifest_path,
    root=ROOT,
    include_csv_shape=True,
)

counts = {
    "table_csv_count": len(list((ROOT / "06_outputs/tables/csv").glob("table_*.csv"))),
    "table_png_count": len(list((ROOT / "06_outputs/tables/png").glob("table_*.png"))),
    "table_svg_count": len(list((ROOT / "06_outputs/tables/svg").glob("table_*.svg"))),
    "figure_png_count": len(list((ROOT / "06_outputs/06_outputs/figures/active/active/png").glob("figure_*.png"))),
    "figure_svg_count": len(list((ROOT / "06_outputs/06_outputs/figures/active/active/svg").glob("figure_*.svg"))),
    "figure_source_csv_count": len(list((ROOT / "06_outputs/06_outputs/figures/active/active/source_data").glob("figure_*.csv"))),
}
assert counts == {
    "table_csv_count": 6,
    "table_png_count": 6,
    "table_svg_count": 6,
    "figure_png_count": 17,
    "figure_svg_count": 17,
    "figure_source_csv_count": 17,
}, counts

forbidden_name_tokens = [
    "charging_pathway", "charging_event_pathway", "charging_demand", "grid_load",
    "potential_charging_event", "route_to_charging_pathway",
]
active_relative_paths = [str(path.relative_to(ROOT)).lower() for path in ROOT.glob("outputs/**/*") if path.is_file()]
for token in forbidden_name_tokens:
    matches = [path for path in active_relative_paths if token in path]
    if matches:
        raise RuntimeError(f"Forbidden active output token {token!r}: {matches}")

integrity = read_csv_checked(
    ROOT / "06_outputs/reports/validation/686_20260711_research_integrity_checks.csv",
    required_columns=["check", "status", "observed"],
    require_nonempty=True,
)
assert integrity["status"].eq("pass").all()
step_status_df = __import__("pandas").DataFrame(STEP_STATUS)
assert step_status_df["status"].isin(["success", "validated_existing"]).all()
write_csv_atomic(step_status_df, ROOT / "06_outputs/reports/validation/notebook_679_20260711_execution_status.csv")

validation_summary = __import__("pandas").DataFrame([
    {
        "validation_status": "success",
        "analysis_run_id": analysis_manifest["analysis_run_id"].iloc[0],
        "python_version": sys.version.split()[0],
        "independent_seed_count": int(statistics["independent_seed_count"].iloc[0]),
        "conditional_evaluation_count": int(statistics["conditional_evaluation_count"].iloc[0]),
        "condition_route_evaluation_count": int(statistics["condition_route_evaluation_count"].iloc[0]),
        "bootstrap_iterations": BOOTSTRAP_ITERATIONS,
        **counts,
        "charging_pathway_generated": False,
        "charging_demand_estimation_generated": False,
        "charging_event_timeline_generated": False,
        "grid_load_estimation_generated": False,
    }
])
write_csv_atomic(validation_summary, ROOT / "06_outputs/reports/validation/681_20260711_final_validation_summary.csv")
final_manifest = generate_output_manifest(
    canonical_dirs,
    manifest_path=final_manifest_path,
    root=ROOT,
    include_csv_shape=True,
)

display(Markdown("## ✅ Validation passed"))
display(step_status_df)
display(validation_summary)
print("Charging pathway generated: False")
print("Charging-demand estimation generated: False")
print("Charging-event timeline generated: False")
print("Grid-load estimation generated: False")

## ✅ Validation passed

,step_name,status,return_code,duration_seconds,stdout_log,stderr_log,warning,error
0,analysis_csv_generation,success,0,83.916986,06_outputs/reports/logs/675_20260711_analysis_csv_generation_stdout.log,06_outputs/reports/logs/674_20260711_analysis_csv_generation_stderr.log,/opt/anaconda3/lib/python3.11/site-packages/pa...,
1,csv_driven_rendering,success,0,14.156987,06_outputs/reports/logs/677_20260711_csv_driven_rendering_stdout.log,06_outputs/reports/logs/676_20260711_csv_driven_rendering_stderr.log,/opt/anaconda3/lib/python3.11/site-packages/pa...,


,validation_status,analysis_run_id,python_version,independent_seed_count,conditional_evaluation_count,condition_route_evaluation_count,bootstrap_iterations,table_csv_count,table_png_count,table_svg_count,figure_png_count,figure_svg_count,figure_source_csv_count,charging_pathway_generated,charging_demand_estimation_generated,charging_event_timeline_generated,grid_load_estimation_generated
0,success,20260711T072949Z,3.11.8,100,2700,8100,1000,6,6,6,17,17,17,False,False,False,False


Charging pathway generated: False
Charging-demand estimation generated: False
Charging-event timeline generated: False
Grid-load estimation generated: False
